In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder \
    .appName("Semana12_Regresion") \
    .getOrCreate()

ruta_datos = "/home/jovyan/work/autotec/luz/Semana 10/modelos/datos_etiquetados_kmeans"

df_clusters = spark.read.parquet(ruta_datos)

print("Datos cargados:", df_clusters.count())
df_clusters.printSchema()
df_clusters.show(5, truncate=False)

Datos cargados: 1955
root
 |-- marca: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- year: long (nullable = true)
 |-- kilometraje: double (nullable = true)
 |-- combustible: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- marca_cat: long (nullable = true)
 |-- modelo_cat: long (nullable = true)
 |-- combustible_cat: long (nullable = true)
 |-- ciudad_cat: long (nullable = true)
 |-- cluster_kmeans: long (nullable = true)
 |-- prediction: long (nullable = true)

+-------+----------------------------+----+-----------+-----------+--------+-------+---------+----------+---------------+----------+--------------+----------+
|marca  |modelo                      |year|kilometraje|combustible|ciudad  |precio |marca_cat|modelo_cat|combustible_cat|ciudad_cat|cluster_kmeans|prediction|
+-------+----------------------------+----+-----------+-----------+--------+-------+---------+----------+---------------+----------+----

In [4]:
assembler_regresion = VectorAssembler(
    inputCols=[
        "marca_cat",
        "modelo_cat",
        "year",
        "kilometraje",
        "combustible_cat",
        "ciudad_cat"
    ],
    outputCol="features_regresion"
)

df_vector_reg = assembler_regresion.transform(df_clusters)

scaler_reg = StandardScaler(
    inputCol="features_regresion",
    outputCol="scaledFeatures_regresion"
)

scaler_model_reg = scaler_reg.fit(df_vector_reg)

df_regresion = scaler_model_reg.transform(df_vector_reg)

df_regresion = df_regresion.withColumnRenamed("precio", "label_precio")

df_regresion = df_regresion.drop("prediction")

train_reg, test_reg = df_regresion.randomSplit([0.7, 0.3], seed=42)

print("Registros entrenamiento:", train_reg.count())
print("Registros prueba:", test_reg.count())

Registros entrenamiento: 1384
Registros prueba: 571


In [5]:
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion",
    labelCol="label_precio",
    maxIter=10
)

lr_reg_model = lr_regresion.fit(train_reg)

predicciones_regresion = lr_reg_model.transform(test_reg)

predicciones_regresion.select(
    "marca",
    "modelo",
    "label_precio",
    "prediction"
).show(10, truncate=False)

+-------+----------------------------+------------+--------------------+
|marca  |modelo                      |label_precio|prediction          |
+-------+----------------------------+------------+--------------------+
|changan|Cs35                        |1.049E7     |1.874717501445377E7 |
|chery  |Tiggo 7                     |1.129E7     |1.7653975396775126E7|
|dodge  |Nitro Slt 4x4 3.7           |9990000.0   |1.043282330861485E7 |
|ducati |Multistrada 1.200 S         |1.298E7     |1.454479517508626E7 |
|great  |H3                          |6990000.0   |1.514860113237524E7 |
|great  |Poer                        |1.799E7     |2.1569166148070216E7|
|great  |Poer Plus                   |2.199E7     |2.024246177500236E7 |
|haval  |H6                          |1.299E7     |1.6718795742233276E7|
|hyundai|Tucson                      |1.849E7     |1.6529001939067721E7|
|peugeot|508 Peugeot 508 2.0 Hdi Auto|1.598E7     |1.574791693784237E7 |
+-------+----------------------------+------------+

In [6]:
evaluator_r2 = RegressionEvaluator(
    labelCol="label_precio",
    predictionCol="prediction",
    metricName="r2"
)

evaluator_rmse = RegressionEvaluator(
    labelCol="label_precio",
    predictionCol="prediction",
    metricName="rmse"
)

r2 = evaluator_r2.evaluate(predicciones_regresion)
rmse = evaluator_rmse.evaluate(predicciones_regresion)

print("=========================================")
print("RESULTADOS REGRESIÓN LINEAL")
print("=========================================")
print(f"R2: {r2 * 100:.2f}%")
print(f"RMSE: ${rmse:,.0f}".replace(",", "."))
print("=========================================")

RESULTADOS REGRESIÓN LINEAL
R2: 11.89%
RMSE: $9.964.438


In [7]:
print("========================================")
print("ECUACIÓN DEL MODELO")
print("========================================")

print(f"Intercepto: {lr_reg_model.intercept:.4f}")

variables = [
    "marca_cat",
    "modelo_cat",
    "year",
    "kilometraje",
    "combustible_cat",
    "ciudad_cat"
]

for nombre, coef in zip(variables, lr_reg_model.coefficients):
    print(f"{nombre}: {coef:.4f}")

ECUACIÓN DEL MODELO
Intercepto: -973630220.0617
marca_cat: -625537.3111
modelo_cat: -742867.9778
year: 1744443.9974
kilometraje: -1232007.1882
combustible_cat: 1296027.4903
ciudad_cat: 421620.2669


Interpretación:

La regresión lineal fue utilizada para estimar el precio de los vehículos a partir de variables como marca, modelo, año de fabricación, kilometraje, tipo de combustible y ciudad.

El modelo obtuvo un R² de 11,89%, lo que indica que solo una parte de la variación de los precios puede ser explicada por las variables consideradas en el análisis. Esto sugiere que existen otros factores relevantes que influyen en el valor de los vehículos y que no están presentes en la base de datos.

El RMSE obtenido fue de aproximadamente $9.964.438, lo que refleja una diferencia considerable entre los precios reales y los precios estimados por el modelo. Por esta razón, las predicciones deben interpretarse como una aproximación general y no como un valor exacto.

Al analizar los coeficientes del modelo, se observa que el año de fabricación tiene una influencia positiva sobre el precio, mientras que el kilometraje presenta una relación negativa, lo que coincide con el comportamiento esperado del mercado automotriz.

En conclusión, la regresión lineal permite identificar tendencias generales en los datos de AutoTec, pero su capacidad predictiva es limitada. Para obtener mejores resultados sería recomendable incorporar nuevas variables o utilizar modelos más avanzados de regresión.
